Lưu MinIO

In [ ]:
!pip install confluent_kafka minio pandas

In [ ]:
!pip install confluent_kafka minio

from confluent_kafka import Consumer, KafkaError
from minio import Minio
import json
import csv
import io
import time
from datetime import datetime

# --- 1. CẤU HÌNH MINIO (Dùng server test play.min.io) ---
MINIO_ENDPOINT = 'play.min.io'
ACCESS_KEY = 'minioadmin'
SECRET_KEY = 'minioadmin'
BUCKET_NAME = 'weather-data-demo'   # Đổi tên bucket chút để tránh trùng với người khác

# --- 2. CẤU HÌNH KAFKA ---
KAFKA_HOST = '18.219.196.163:9092'  # IP Kafka của bạn
TOPIC_NAME = 'weather_multi_city'
GROUP_ID = 'minio-saver-group-secure-v2' # Đổi Group ID để đọc lại từ đầu

# --- 3. DANH SÁCH 17 CỘT (HEADER) ---
CSV_HEADERS = [
    "sensor_id", "city", "region",           # Định danh
    "timestamp",                             # Thời gian
    "temperature", "humidity", "apparent_temperature",
    "weather_code", "pressure", "rain",
    "wind_speed", "wind_direction", "wind_gusts",
    "solar_radiation", "visibility", "cloud_cover", "is_day"
]

def main():
    print("⏳ Đang kết nối đến MinIO (HTTPS)...")

    # A. KẾT NỐI MINIO
    # QUAN TRỌNG: secure=True để dùng cổng 443 (HTTPS), tránh bị chặn cổng 80
    minio_client = Minio(
        MINIO_ENDPOINT,
        access_key=ACCESS_KEY,
        secret_key=SECRET_KEY,
        secure=True
    )

    # Tạo bucket nếu chưa có
    try:
        if not minio_client.bucket_exists(BUCKET_NAME):
            minio_client.make_bucket(BUCKET_NAME)
            print(f"📁 Đã tạo Bucket mới: {BUCKET_NAME}")
        else:
            print(f"📁 Đã tìm thấy Bucket: {BUCKET_NAME}")
    except Exception as e:
        print(f"❌ Lỗi kết nối MinIO: {e}")
        return # Dừng chương trình nếu không kết nối được MinIO

    # B. KẾT NỐI KAFKA CONSUMER
    conf = {
        'bootstrap.servers': KAFKA_HOST,
        'group.id': GROUP_ID,
        'auto.offset.reset': 'earliest'
    }
    consumer = Consumer(conf)
    consumer.subscribe([TOPIC_NAME])

    print(f"🎧 Đang lắng nghe từ Topic: {TOPIC_NAME}...")
    print(f"📦 Đang chờ gom đủ 4 bản tin để đóng gói CSV...")
    print("-" * 50)

    buffer = [] # Nơi chứa tạm dữ liệu

    try:
        while True:
            msg = consumer.poll(1.0)

            if msg is None:
                continue
            if msg.error():
                print(f"⚠️ Lỗi Kafka: {msg.error()}")
                continue

            # 1. Giải mã tin nhắn
            try:
                data_str = msg.value().decode('utf-8')
                data_json = json.loads(data_str)
                buffer.append(data_json)

                print(f" + Nhận: {data_json.get('city', 'Unknown')} ({len(buffer)}/4)")
            except Exception as e:
                print(f"⚠️ Lỗi giải mã dữ liệu: {e}")
                continue

            # 2. KIỂM TRA: Nếu đủ 4 tin (tức là đủ 4 thành phố trong đợt này)
            if len(buffer) >= 4:
                # Tạo tên file theo thời gian thực
                now_str = datetime.now().strftime("%Y%m%d_%H%M%S")
                file_name = f"weather_data_{now_str}.csv"

                # 3. CHUYỂN JSON -> CSV (Trong bộ nhớ RAM)
                csv_buffer = io.StringIO()
                writer = csv.DictWriter(csv_buffer, fieldnames=CSV_HEADERS)

                writer.writeheader()     # Viết dòng tiêu đề
                writer.writerows(buffer) # Viết dữ liệu

                # Chuẩn bị đẩy lên MinIO
                csv_bytes = csv_buffer.getvalue().encode('utf-8')
                csv_stream = io.BytesIO(csv_bytes)

                # 4. UPLOAD LÊN MINIO
                try:
                    minio_client.put_object(
                        BUCKET_NAME,
                        file_name,
                        csv_stream,
                        length=len(csv_bytes),
                        content_type='application/csv'
                    )
                    print(f"✅ ĐÃ LƯU THÀNH CÔNG: {file_name} lên MinIO")
                    print("-" * 50)
                except Exception as e:
                    print(f"❌ Lỗi khi upload file lên MinIO: {e}")

                # Reset bộ nhớ đệm
                buffer = []

    except KeyboardInterrupt:
        print("\n🛑 Đã dừng chương trình.")
    finally:
        consumer.close()

if __name__ == "__main__":
    main()

⏳ Đang kết nối đến MinIO (HTTPS)...
📁 Đã tìm thấy Bucket: weather-data-demo
🎧 Đang lắng nghe từ Topic: weather_multi_city...
📦 Đang chờ gom đủ 4 bản tin để đóng gói CSV...
--------------------------------------------------
 + Nhận: Ha Noi (1/4)
 + Nhận: Hai Phong (2/4)
 + Nhận: Da Nang (3/4)
 + Nhận: Can Tho (4/4)
✅ ĐÃ LƯU THÀNH CÔNG: weather_data_20260130_160102.csv lên MinIO
--------------------------------------------------
